In [0]:
# ==============================================================================
# 1. DEFINICIÓN DE VARIABLES (¡REEMPLAZA AQUÍ!)
# ==============================================================================
storage_account_name = "datalakeexperto2025"
storage_account_key = "xxxsecretxxx"           # acces key de Azure
container_name = "raw-data"
file_name = "youtube_shorts_tiktok_trends_2025.csv_ML.csv" # O el nombre exacto de tu archivo (ej: datos.json)


# ==============================================================================
# 2. CONFIGURACIÓN DE LA CONEXIÓN (Autenticación)
# ==============================================================================
# Establece la clave de acceso para el Storage Account.
# Esto es esencial para que Spark sepa que tiene permiso de lectura.
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Definimos la ruta completa de ADLS Gen2 (abfss:// es el protocolo)
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"


# ==============================================================================
# 3. LECTURA DEL ARCHIVO EN UN DATAFRAME (DF)
# ==============================================================================
# Si es CSV, usar este formato
if file_name.endswith(".csv"):
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(adls_path)

# Si es JSON, usar este formato
elif file_name.endswith((".json", ".jsonl")):
    df_raw = spark.read.json(adls_path)

else:
    print("Formato de archivo no soportado. Edita el código de lectura.")
    
    
# ==============================================================================
# 4. PRUEBA DE ÉXITO
# ==============================================================================
print(f"Ruta de archivo leída: {adls_path}")
print("\nMostrando las primeras 5 filas (DF creado):")
df_raw.show(5)

print("\nEsquema (Schema) del DataFrame:")
df_raw.printSchema()

Ruta de archivo leída: abfss://raw-data@datalakeexperto2025.dfs.core.windows.net/youtube_shorts_tiktok_trends_2025.csv_ML.csv

Mostrando las primeras 5 filas (DF creado):
+-----------+--------+--------+--------+--------+--------------+------------+------------+---------+------------------+-------------------+------------------+------------------+-------------------+--------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------+------------------------+-------------------------+------------+----------+------------+------------+------------------+----------------+----------------+----------------------------+---------------------+
|trend_label|platform|  region|language|category|traffic_source|device_brand|creator_tier|title_len|     text_richness|          like_rate|      comment_rate|        share_rate|      like_rate_log|    comment_rate_log|      share_rate_log|      views_per_day|       likes_per_day| 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, round, avg, sum

# 1. Limpieza y Filtrado Básico: 
# Excluimos las filas donde la tasa de 'like_rate' sea nula o cero, 
# ya que no aportan valor para el análisis de rendimiento.
df_filtered = df_raw.filter(
    (col("like_rate").isNotNull()) & (col("like_rate") > 0)
)

print(f"Filas originales: {df_raw.count()}")
print(f"Filas después del filtro: {df_filtered.count()}")

Filas originales: 50000
Filas después del filtro: 49985


In [0]:
# 2. Creación de una Métrica de Rendimiento (Key Performance Indicator - KPI):
# Creamos un score que combina la popularidad relativa (like_rate) y la riqueza del contenido (text_richness).
df_transformed = df_filtered.withColumn(
    "engagement_score", 
    round(
        (col("like_rate") * 100) + col("text_richness"), 
        3
    )
).withColumnRenamed(
    "title_len", 
    "longitud_titulo"
)

# Mostramos el cambio en el esquema (solo las columnas relevantes)
print("\nDataFrame con nueva métrica y columna renombrada:")
df_transformed.select("like_rate", "text_richness", "engagement_score", "longitud_titulo").show(5, truncate=False)


DataFrame con nueva métrica y columna renombrada:
+-------------------+------------------+----------------+---------------+
|like_rate          |text_richness     |engagement_score|longitud_titulo|
+-------------------+------------------+----------------+---------------+
|0.2899972462945625 |1.9999980000020001|31.0            |2              |
|0.11299877656377774|1.9999980000020001|13.3            |2              |
|0.1333104033242175 |1.9999980000020001|15.331          |2              |
|0.24446353095248707|1.9999980000020001|26.446          |2              |
|0.23294243384089763|1.9999980000020001|25.294          |2              |
+-------------------+------------------+----------------+---------------+
only showing top 5 rows


In [0]:
# 3. Agregación (Agrupación):
# Calculamos métricas resumidas por las dimensiones "platform" y "region"
df_aggregated = df_transformed.groupBy("platform", "region").agg(
    # Calcula la tasa promedio de 'views_per_day'
    avg("views_per_day").alias("avg_views_per_day"),
    
    # Encuentra la puntuación máxima de interacción
    F.max("engagement_score").alias("max_engagement_score"),
    
    # Cuenta cuántos videos se analizaron en este grupo
    F.count("*").alias("total_videos")
).orderBy(col("avg_views_per_day").desc())

# 4. Prueba Final: Mostrar el resultado del ELT
print("\nResultado de la Agregación (Métricas por Plataforma y Región):")
df_aggregated.show(10, truncate=False)


Resultado de la Agregación (Métricas por Plataforma y Región):
+--------+--------+-------------------+--------------------+------------+
|platform|region  |avg_views_per_day  |max_engagement_score|total_videos|
+--------+--------+-------------------+--------------------+------------+
|tiktok  |Oceania |0.12636338690132687|89.309              |1581        |
|tiktok  |Asia    |0.12587972836554276|83.249              |7280        |
|tiktok  |MENA    |0.12570628314500706|72.623              |2097        |
|tiktok  |Americas|0.12552560834569584|86.343              |7216        |
|tiktok  |Africa  |0.12542354096343591|72.119              |2053        |
|tiktok  |Europe  |0.12524739296337267|74.688              |5684        |
|youtube |Africa  |0.10392202689886959|69.828              |1987        |
|youtube |Asia    |0.10327984967866115|75.625              |6604        |
|youtube |Europe  |0.1032338550379108 |71.527              |5304        |
|youtube |MENA    |0.1031860020188451 |73.163   

In [0]:
# Definición de la ruta de salida en ADLS Gen2, bajo el contenedor 'raw-data'
# Creamos una carpeta 'curated-data' para los datos limpios.
storage_account_name = "datalakeexperto2025"
container_name = "raw-data" # Usamos el mismo contenedor
delta_output_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/curated-data/platform_summary"

# Escribir el DataFrame 'df_aggregated' en formato Delta.
# mode("overwrite") es solo para el laboratorio. En producción se usaría 'append' o 'merge'.
print(f"Escribiendo datos agregados en formato Delta en: {delta_output_path}")

df_aggregated.write.format("delta") \
    .mode("overwrite") \
    .save(delta_output_path)

print("¡Escritura en Delta Lake completada!")

Escribiendo datos agregados en formato Delta en: abfss://raw-data@datalakeexperto2025.dfs.core.windows.net/curated-data/platform_summary
¡Escritura en Delta Lake completada!


In [0]:
# Definición de la ruta donde guardamos la tabla Delta
storage_account_name = "datalakeexperto2025" 
container_name = "raw-data" 
delta_table_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/curated-data/platform_summary"

# Leemos la tabla Delta completa
# Delta Lake solo necesita la ruta de la carpeta raíz
print(f"Leyendo la tabla Delta desde: {delta_table_path}")

df_delta_read = spark.read.format("delta").load(delta_table_path)

# Verificación de que la lectura fue exitosa
print("\nLectura exitosa. Mostrando las primeras filas del DataFrame recuperado:")
df_delta_read.show(5)

print("\nEsquema de la tabla Delta:")
df_delta_read.printSchema()

Leyendo la tabla Delta desde: abfss://raw-data@datalakeexperto2025.dfs.core.windows.net/curated-data/platform_summary

Lectura exitosa. Mostrando las primeras filas del DataFrame recuperado:
+--------+--------+-------------------+--------------------+------------+
|platform|  region|  avg_views_per_day|max_engagement_score|total_videos|
+--------+--------+-------------------+--------------------+------------+
|  tiktok| Oceania|0.12636338690132687|              89.309|        1581|
|  tiktok|    Asia|0.12587972836554276|              83.249|        7280|
|  tiktok|    MENA|0.12570628314500706|              72.623|        2097|
|  tiktok|Americas|0.12552560834569584|              86.343|        7216|
|  tiktok|  Africa|0.12542354096343591|              72.119|        2053|
+--------+--------+-------------------+--------------------+------------+
only showing top 5 rows

Esquema de la tabla Delta:
root
 |-- platform: string (nullable = true)
 |-- region: string (nullable = true)
 |-- avg